In [1]:
# ============================================================
# 0. IMPORT LIBRARY
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Install factor_analyzer if not already installed
!pip install factor_analyzer

from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

from google.colab import drive
# mount drive
drive.mount('/content/drive')

# load data
inflasi = pd.read_excel("/content/drive/My Drive/Data Inflasi All.xlsx")
birate = pd.read_excel("/content/drive/My Drive/Data BI Rate All.xlsx")
kurs = pd.read_excel("/content/drive/My Drive/Data Kurs All.xlsx")
PDB = pd.read_excel("/content/drive/My Drive/Data PDB All.xlsx")

# Menghitung Log Return
kurs['Data Kurs'] = np.log(kurs['Data Kurs'] / kurs['Data Kurs'].shift(1))
PDB['Data GDP'] = np.log(PDB['Data GDP'] / PDB['Data GDP'].shift(1))

inflasi.head()
birate.head()
kurs.head()
PDB.head()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for factor_analyzer: filename=factor_analyzer-0.5.1-py2.py3-none-any.whl size=42655 sha256=c6832c903ff07395b67319848116b389043049536f36b020aa9e6e11dc6e1dfe
  Stored in directory: /root/.cache/pip/wheels/a2/af/06/f4d4ed4d9d714fda437fb1583629417319603c2266e7b233cc
Successfully built factor_analyzer
Mounted at /content/drive


,Tanggal,Data GDP
0,2010-01-01,NaN
1,2010-02-01,0.013969
2,2010-03-01,0.012452
3,2010-04-01,0.013607
4,2010-05-01,0.012994


In [2]:
from functools import reduce

# 1. Pastikan kolom Tanggal di setiap dataset sudah bertipe datetime
# Ini sangat penting agar proses penggabungan (matching) tidak error
inflasi['Tanggal'] = pd.to_datetime(inflasi['Tanggal'])
birate['Tanggal'] = pd.to_datetime(birate['Tanggal'])
kurs['Tanggal'] = pd.to_datetime(kurs['Tanggal'])
PDB['Tanggal'] = pd.to_datetime(PDB['Tanggal'])

# 2. Masukkan semua dataframe ke dalam satu list
data_frames = [inflasi, birate, kurs, PDB]

# 3. Gabungkan semuanya secara berurutan menggunakan merge
# how='outer' digunakan agar tidak ada data tanggal yang hilang jika ada perbedaan periode
makro = reduce(lambda left, right: pd.merge(left, right, on='Tanggal', how='outer'), data_frames)
makro = makro.sort_values('Tanggal').reset_index(drop=True)
makro = makro.dropna().reset_index(drop=True)

makro_2010_2024 = makro[makro['Tanggal'].dt.year <= 2024]
makro_2025 = makro[makro['Tanggal'].dt.year == 2025]

# 5. Cek hasilnya
print(makro_2010_2024.head())
print(makro_2025.head())

     Tanggal  Data Inflasi  Data BI Rate  Data Kurs  Data GDP
0 2010-02-01        0.0381         0.065  -0.001391  0.013969
1 2010-03-01        0.0343         0.065  -0.026810  0.012452
2 2010-04-01        0.0391         0.065  -0.008562  0.013607
3 2010-05-01        0.0416         0.065   0.017870  0.012994
4 2010-06-01        0.0505         0.065  -0.012613  0.013252
       Tanggal  Data Inflasi  Data BI Rate  Data Kurs  Data GDP
179 2025-01-01      0.015993      0.059212  -0.012967  0.005942
180 2025-02-01      0.016537      0.058420  -0.006634  0.005907
181 2025-03-01      0.017214      0.057625  -0.003854  0.005305
182 2025-04-01      0.017932      0.056833  -0.003366  0.007328
183 2025-05-01      0.018677      0.056048  -0.002587  0.007041


In [3]:
# ============================================================
# 2. COPY DATASET
# ============================================================

train_df = makro_2010_2024.copy()
future_df = makro_2025.copy()

# ============================================================
# 3. TENTUKAN KOLOM
# ============================================================

date_col = "Tanggal"
macro_cols = ["Data Inflasi", "Data BI Rate", "Data Kurs", "Data GDP"]

# ============================================================
# 4. RAPIIKAN TIPE DATA
# ============================================================

train_df[date_col] = pd.to_datetime(train_df[date_col])
future_df[date_col] = pd.to_datetime(future_df[date_col])

for col in macro_cols:
    train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
    future_df[col] = pd.to_numeric(future_df[col], errors="coerce")

train_df = train_df.dropna(subset=macro_cols).reset_index(drop=True)
future_df = future_df.dropna(subset=macro_cols).reset_index(drop=True)

print("=== DATA TRAIN 2010-2024 ===")
print(train_df.head())
print("\n=== DATA 2025 ===")
print(future_df.head())

=== DATA TRAIN 2010-2024 ===
     Tanggal  Data Inflasi  Data BI Rate  Data Kurs  Data GDP
0 2010-02-01        0.0381         0.065  -0.001391  0.013969
1 2010-03-01        0.0343         0.065  -0.026810  0.012452
2 2010-04-01        0.0391         0.065  -0.008562  0.013607
3 2010-05-01        0.0416         0.065   0.017870  0.012994
4 2010-06-01        0.0505         0.065  -0.012613  0.013252

=== DATA 2025 ===
     Tanggal  Data Inflasi  Data BI Rate  Data Kurs  Data GDP
0 2025-01-01      0.015993      0.059212  -0.012967  0.005942
1 2025-02-01      0.016537      0.058420  -0.006634  0.005907
2 2025-03-01      0.017214      0.057625  -0.003854  0.005305
3 2025-04-01      0.017932      0.056833  -0.003366  0.007328
4 2025-05-01      0.018677      0.056048  -0.002587  0.007041


In [4]:
# ============================================================
# 5. UJI VIF (PAKAI DATA ASLI)
# ============================================================

X_vif = train_df[macro_cols].copy()

vif_data = pd.DataFrame()
vif_data["Variabel"] = macro_cols
vif_data["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

print("\n================ HASIL UJI VIF ================")
print(vif_data)


================ HASIL UJI VIF ================
       Variabel        VIF
0  Data Inflasi  10.783002
1  Data BI Rate  10.938344
2     Data Kurs   1.063049
3      Data GDP   1.287221


In [5]:
# ============================================================
# 6. STANDARDISASI DATA TRAIN
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(train_df[macro_cols])

X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=macro_cols
)

# simpan mean dan std training
scaler_info = pd.DataFrame({
    "Variabel": macro_cols,
    "Mean_Train": scaler.mean_,
    "Std_Train": scaler.scale_
})

print("\n================ HASIL STANDARDISASI TRAIN ================")
print(X_train_scaled_df.head())

print("\n================ MEAN DAN STD TRAIN ================")
print(scaler_info)


================ HASIL STANDARDISASI TRAIN ================
   Data Inflasi  Data BI Rate  Data Kurs  Data GDP
0     -0.173326      0.650375  -0.193579  1.247370
1     -0.379474      0.650375  -1.305849  1.056734
2     -0.119076      0.650375  -0.507369  1.201920
3      0.016548      0.650375   0.649251  1.124916
4      0.499369      0.650375  -0.684626  1.157339

================ MEAN DAN STD TRAIN ================
       Variabel  Mean_Train  Std_Train
0  Data Inflasi    0.041295   0.018433
1  Data BI Rate    0.056872   0.012498
2     Data Kurs    0.003033   0.022853
3      Data GDP    0.004040   0.007960


In [6]:
# ============================================================
# 7. UJI KMO (PAKAI DATA YANG SUDAH DISTANDARDISASI)
# ============================================================

kmo_all, kmo_model = calculate_kmo(X_train_scaled_df)

kmo_table = pd.DataFrame({
    "Variabel": macro_cols,
    "KMO": kmo_all
})

print("\n================ HASIL UJI KMO PER VARIABEL ================")
print(kmo_table)

print("\nKMO Keseluruhan =", round(kmo_model, 4))


================ HASIL UJI KMO PER VARIABEL ================
       Variabel       KMO
0  Data Inflasi  0.498394
1  Data BI Rate  0.496844
2     Data Kurs  0.438774
3      Data GDP  0.573027

KMO Keseluruhan = 0.4969


In [7]:
# ============================================================
# 8. UJI BARTLETT (PAKAI DATA YANG SUDAH DISTANDARDISASI)
# ============================================================

chi_square_value, p_value = calculate_bartlett_sphericity(X_train_scaled_df)

print("\n================ HASIL BARTLETT TEST ================")
print("Chi-Square :", round(chi_square_value, 4))
print("p-value    :", p_value)


================ HASIL BARTLETT TEST ================
Chi-Square : 117.7569
p-value    : 4.8200433705440125e-23


In [8]:
# ============================================================
# 9. INTERPRETASI AWAL KELAYAKAN PCA
# ============================================================

print("\n================ INTERPRETASI ================")

# interpretasi VIF
for _, row in vif_data.iterrows():
    var = row["Variabel"]
    val = row["VIF"]
    if val < 5:
        print(f"{var}: VIF = {val:.4f} -> aman")
    elif val < 10:
        print(f"{var}: VIF = {val:.4f} -> multikolinearitas sedang")
    else:
        print(f"{var}: VIF = {val:.4f} -> multikolinearitas tinggi")

# interpretasi KMO
if kmo_model >= 0.90:
    print(f"KMO keseluruhan = {kmo_model:.4f} -> sangat baik untuk PCA")
elif kmo_model >= 0.80:
    print(f"KMO keseluruhan = {kmo_model:.4f} -> baik untuk PCA")
elif kmo_model >= 0.70:
    print(f"KMO keseluruhan = {kmo_model:.4f} -> cukup baik untuk PCA")
elif kmo_model >= 0.60:
    print(f"KMO keseluruhan = {kmo_model:.4f} -> cukup / masih dapat diterima")
elif kmo_model >= 0.50:
    print(f"KMO keseluruhan = {kmo_model:.4f} -> marginal")
else:
    print(f"KMO keseluruhan = {kmo_model:.4f} -> kurang layak untuk PCA")

# interpretasi Bartlett
if p_value < 0.05:
    print(f"Bartlett p-value = {p_value:.6f} -> signifikan, data layak untuk PCA")
else:
    print(f"Bartlett p-value = {p_value:.6f} -> tidak signifikan, data kurang layak untuk PCA")


================ INTERPRETASI ================
Data Inflasi: VIF = 10.7830 -> multikolinearitas tinggi
Data BI Rate: VIF = 10.9383 -> multikolinearitas tinggi
Data Kurs: VIF = 1.0630 -> aman
Data GDP: VIF = 1.2872 -> aman
KMO keseluruhan = 0.4969 -> kurang layak untuk PCA
Bartlett p-value = 0.000000 -> signifikan, data layak untuk PCA


In [9]:
# ============================================================
# 10. PCA PENUH UNTUK MELIHAT EXPLAINED VARIANCE
# ============================================================

pca_full = PCA()
pca_full.fit(X_train_scaled)

explained_var = pca_full.explained_variance_ratio_
cum_explained_var = np.cumsum(explained_var)

explained_variance_table = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(explained_var))],
    "Explained_Variance_Ratio": explained_var,
    "Cumulative_Explained_Variance": cum_explained_var
})

print("\n================ EXPLAINED VARIANCE ================")
print(explained_variance_table)


================ EXPLAINED VARIANCE ================
    PC  Explained_Variance_Ratio  Cumulative_Explained_Variance
0  PC1                  0.430434                       0.430434
1  PC2                  0.274572                       0.705006
2  PC3                  0.217048                       0.922055
3  PC4                  0.077945                       1.000000


In [10]:
# ============================================================
# 11. PILIH JUMLAH KOMPONEN
#     ambil minimum PC dengan cumulative >= 90%
# ============================================================

n_components_selected = np.argmax(cum_explained_var >= 0.90) + 1
print(f"\nJumlah PC terpilih = {n_components_selected}")


Jumlah PC terpilih = 3


In [11]:
# ============================================================
# 12. FIT PCA AKHIR
# ============================================================

pca = PCA(n_components=n_components_selected)
X_train_pca = pca.fit_transform(X_train_scaled)

In [12]:
# ============================================================
# 13. LOADINGS PCA
# ============================================================

loadings = pd.DataFrame(
    pca.components_.T,
    index=macro_cols,
    columns=[f"PC{i+1}" for i in range(n_components_selected)]
)

print("\n================ LOADINGS PCA ================")
print(loadings)


================ LOADINGS PCA ================
                   PC1       PC2       PC3
Data Inflasi  0.689318 -0.143645 -0.060489
Data BI Rate  0.676231 -0.222901  0.088769
Data Kurs     0.197421  0.678160 -0.698599
Data GDP      0.169048  0.685406  0.707405


In [13]:
# ============================================================
# 14. EIGENVALUE
# ============================================================

eigenvalues = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(pca_full.explained_variance_))],
    "Eigenvalue": pca_full.explained_variance_
})

print("\n================ EIGENVALUE ================")
print(eigenvalues)


================ EIGENVALUE ================
    PC  Eigenvalue
0  PC1    1.731410
1  PC2    1.104458
2  PC3    0.873070
3  PC4    0.313533


In [14]:
# ============================================================
# 15. PC SCORES 2010-2024
# ============================================================

pc_scores_2010_2024 = pd.DataFrame(
    X_train_pca,
    columns=[f"PC{i+1}" for i in range(n_components_selected)]
)

pc_scores_2010_2024.insert(0, date_col, train_df[date_col].values)

print("\n================ PC SCORES 2010-2024 ================")
print(pc_scores_2010_2024.head())


================ PC SCORES 2010-2024 ================
     Tanggal       PC1       PC2       PC3
0 2010-02-01  0.492976  0.603605  1.085846
1 2010-03-01  0.099063 -0.251742  1.740489
2 2010-04-01  0.460739  0.351861  1.269627
3 2010-05-01  0.769551  1.063974  0.398937
4 2010-06-01  0.844514  0.112261  1.324513


In [15]:
# ============================================================
# 16. STANDARDISASI DATA 2025 DENGAN SCALER TRAIN
# ============================================================

X_future_scaled = scaler.transform(future_df[macro_cols])

X_future_scaled_df = pd.DataFrame(
    X_future_scaled,
    columns=macro_cols
)

print("\n================ DATA 2025 SETELAH STANDARDISASI ================")
print(X_future_scaled_df.head())


================ DATA 2025 SETELAH STANDARDISASI ================
   Data Inflasi  Data BI Rate  Data Kurs  Data GDP
0     -1.372640      0.187306  -0.700085  0.238913
1     -1.343105      0.123871  -0.422971  0.234503
2     -1.306372      0.060316  -0.301359  0.158939
3     -1.267438     -0.003052  -0.280002  0.413012
4     -1.226990     -0.065916  -0.245909  0.376937


In [16]:
# ============================================================
# 17. PROYEKSI DATA 2025 KE PCA TRAIN
# ============================================================

X_future_pca = pca.transform(X_future_scaled)

pc_scores_2025 = pd.DataFrame(
    X_future_pca,
    columns=[f"PC{i+1}" for i in range(n_components_selected)]
)

pc_scores_2025.insert(0, date_col, future_df[date_col].values)

print("\n================ PC SCORES 2025 ================")
print(pc_scores_2025.head())


================ PC SCORES 2025 ================
     Tanggal       PC1       PC2       PC3
0 2025-01-01 -0.917347 -0.155596  0.757743
1 2025-02-01 -0.885922  0.039207  0.553615
2 2025-03-01 -0.892345  0.078777  0.407338
3 2025-04-01 -0.861191  0.275936  0.564171
4 2025-05-01 -0.875188  0.282533  0.506807


In [17]:
# ============================================================
# 18. GABUNGKAN PC SCORES
# ============================================================

pc_all = pd.concat([pc_scores_2010_2024, pc_scores_2025], axis=0).reset_index(drop=True)

print("\n================ GABUNGAN PC SCORES ================")
print(pc_all.head())
print(pc_all.tail())


================ GABUNGAN PC SCORES ================
     Tanggal       PC1       PC2       PC3
0 2010-02-01  0.492976  0.603605  1.085846
1 2010-03-01  0.099063 -0.251742  1.740489
2 2010-04-01  0.460739  0.351861  1.269627
3 2010-05-01  0.769551  1.063974  0.398937
4 2010-06-01  0.844514  0.112261  1.324513
       Tanggal       PC1       PC2       PC3
186 2025-08-01 -0.946752  0.174222  0.260531
187 2025-09-01 -0.958079  0.187437  0.242736
188 2025-10-01 -0.964994  0.217743  0.246819
189 2025-11-01 -0.973168  0.241951  0.247999
190 2025-12-01 -0.988673  0.235223  0.219998


In [ ]:
path_simpan = '/content/drive/My Drive/Data PC All.xlsx'

# Simpan ke Excel
pc_all.to_excel(path_simpan, index=False)